**BRONZE_INGESTION_LAYER **

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS Clinical_Trials_Drug_data

In [0]:
%sql 
USE CATALOG Clinical_Trials_Drug_data;

In [0]:
%sql
show schemas;

In [0]:
%sql
create schema if not exists bronze;

In [0]:
%sql
create schema if not exists silver;

In [0]:
%sql
create schema if not exists gold;

In [0]:
%sql
show schemas

In [0]:
%sql
show tables;

**READING TABLES FROM LANDING FOLDER ADLS GEN2**

In [0]:
%python
drug_df = spark.read.table("clinical_trials_drug_data.default.drug_info_data")
outcomes_df = spark.read.table("clinical_trials_drug_data.default.outcomes_data")
patient_df = spark.read.table("clinical_trials_drug_data.default.patient_info_data")
timeline_df = spark.read.table("clinical_trials_drug_data.default.timeline_data")
trial_df = spark.read.table("clinical_trials_drug_data.default.trial_info_data")

In [0]:
%python
drug_df.display()

In [0]:
%python
display(drug_df.count())

In [0]:
display(outcomes_df.count())

In [0]:
outcomes_df.show()

In [0]:
%python
display(patient_df.count())

In [0]:
patient_df.show()

In [0]:
display(trial_df.count())

In [0]:
%python
timeline_df.show()

In [0]:
display(trial_df.count())

In [0]:
trial_df.show()

**WRITING BACK TO BRONZE LAYER FOLDER IN ADLS GEN2**

In [0]:
drug_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.bronze.drug_info_data")
outcomes_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.bronze.outcomes_data")
patient_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.bronze.patient_data")
timeline_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.bronze.timeline_data")
trial_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.bronze.trial_data")

**pipeline_logs_Bronze**

In [0]:
from datetime import datetime

start_time = datetime.now()

trial_df = spark.read.table("clinical_trials_drug_data.default.trial_info_data")

In [0]:
from pyspark.sql import Row

log_data = [Row(
    pipeline_name="clinical_pipeline_bronze",
    layer_name="bronze",
    table_name="trial_info",
    record_count=trial_df.count(),
    status="SUCCESS",
    start_time=str(start_time),
    end_time=str(datetime.now()),
    error_message=""
)]

spark.createDataFrame(log_data) \
    .write.format("delta") \
    .mode("append") \
    .saveAsTable("clinical_trials_drug_data.bronze.pipeline_logs")

In [0]:
df = spark.read.table("clinical_trials_drug_data.bronze.pipeline_logs")
df.display()